# Story 1 — The Schema Builder
**Goal:** Join the Orders, Reviews, and Customers tables into a single master dataset without duplicating rows.

**Input:** Raw CSVs from `../data/`  
**Output:** `../data/01_master.parquet`

---

In [ ]:
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

print('Libraries loaded.')

## Step 1 — Load the raw CSVs

In [ ]:
orders    = pd.read_csv('../data/olist_orders_dataset.csv')
reviews   = pd.read_csv('../data/olist_order_reviews_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
products  = pd.read_csv('../data/olist_products_dataset.csv')
items     = pd.read_csv('../data/olist_order_items_dataset.csv')
transl    = pd.read_csv('../data/product_category_name_translation.csv', encoding='utf-8-sig')

print(f'orders:    {orders.shape[0]:,} rows, {orders.shape[1]} cols')
print(f'reviews:   {reviews.shape[0]:,} rows, {reviews.shape[1]} cols')
print(f'customers: {customers.shape[0]:,} rows, {customers.shape[1]} cols')
print(f'products:  {products.shape[0]:,} rows, {products.shape[1]} cols')
print(f'items:     {items.shape[0]:,} rows, {items.shape[1]} cols')
print(f'transl:    {transl.shape[0]:,} rows  →  columns: {list(transl.columns)}')

## Step 2 — Check cardinality before joining
The reviews table can have **multiple reviews per order**, which would inflate row counts if we join naively.

In [ ]:
reviews_per_order = reviews.groupby('order_id')['review_id'].count()
print('How many reviews does one order have?')
print(reviews_per_order.value_counts().rename_axis('reviews_count').reset_index(name='orders').to_string(index=False))

As expected — some orders have 2 or 3 reviews. We deduplicate by keeping the **earliest** review per order.

In [ ]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews_dedup = (
    reviews
    .sort_values('review_creation_date')
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score', 'review_creation_date']]
)
print(f'Reviews before dedup: {len(reviews):,}')
print(f'Reviews after dedup:  {len(reviews_dedup):,}  (must be ≤ {len(orders):,} orders)')

## Step 3 — Build the master dataset
Join sequence:  
`orders` → `reviews` (on `order_id`) → `customers` (on `customer_id`)

In [ ]:
master = (
    orders
    .merge(reviews_dedup[['order_id', 'review_score']], on='order_id', how='left')
    .merge(customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left')
)

print(f'master shape: {master.shape}')
print(f'orders shape: {orders.shape}')
master.head(5)

## Step 4 — Row integrity check
The master dataset must have **exactly the same number of rows** as the original orders table.

In [ ]:
assert len(master) == len(orders), f'Row count mismatch! master={len(master)}, orders={len(orders)}'
print(f'✅ Row integrity confirmed: {len(master):,} rows — no duplication.')

print(f'\nMissing review scores: {master["review_score"].isna().sum():,} orders had no review')
print(f'Missing customer state: {master["customer_state"].isna().sum():,} orders had no location')

## Step 5 — Save for next notebook

In [ ]:
master.to_parquet('../data/01_master.parquet', index=False)
print('Saved → ../data/01_master.parquet')
print('\nNext: run 02_delay_calculator.ipynb')